# ARC Qwen38 NVFP4 56K 32K Shuffle

Qwen3.8-27B fine-tuned on its own verified ARC reasoning traces (rejection-sampling self-distillation: 1,036 correct traces from the 1,000 ARC-AGI-2 training tasks and the public evaluation tasks minus a 30-task holdout; LoRA r32, 1 epoch, merged, block-FP8 export in the official Qwen3.8-27B-FP8 layout).

Runner = the exact 26.81 runner with per-attempt temperature. Changes: **64K** thinking cap on both attempts (context 98304), T0.6 on both attempts, model = private dataset thangnm1/qwen38-27b-arc-rft-v1-fp8.

**Single-request attempts** (ARC_SINGLE_REQUEST=1): think + close + answer in one request; only attempts that hit the cap are force-closed and answered with a second request.

 Server: --enable-batch-sharded-sampling, spec {"use_local_argmax_reduction": true}, ARC_SPEC_EXTRA={"use_local_argmax_reduction":true}.

Capped attempts close inside their own request (vLLM thinking_token_budget + reasoning_end_str, patched sampler; the client-side forced close stays as fallback). Server: ARC_PREVIEW_SECONDS=2700 ARC_THINK_BUDGET=56000 ARC_THINK_BUDGET2=32000 ARC_PROGRAMS=1 ARC_EXTRA_ATTEMPTS=1 ARC_REPAIR_THINK=24000,16000 ARC_DEADLINE_CAP=1.

Model: thangnm1/qwen38-27b-arc-rft-v1-nvfp4 (RFT v1 NVFP4). attempt_1 56K thinking cap, attempt_2 32K on a view with the training examples in a different deterministic order (ARC_AUG2=shuffle); ARC_DEADLINE_CAP=1 lowers the cap of attempts that would not finish before the hard stop; crash-safe program checks; attempt_1 for every input first, then attempt_2 (odd inputs first), repairs, extras.


In [1]:
import os, time, subprocess
os.environ['ARC_NOTEBOOK_STARTED_MONOTONIC'] = str(time.monotonic())
os.environ.setdefault('ARC_PREVIEW_SECONDS', '14400')
os.environ.setdefault('ARC_RERUN_SECONDS', '43200')  # full 12 h; the watchdog stops at 11:56
# Kaggle mounts inputs as /kaggle/input/competitions/<slug> or, in the older layout, /kaggle/input/<slug>: pin the real one
import glob as _glob
_comp = sorted(_glob.glob('/kaggle/input/**/arc-agi_test_challenges.json', recursive=True),
               key=lambda p: ('arc-prize-2026-arc-agi-2' not in p, len(p)))
if _comp and not os.getenv('ARC_COMP_DIR'):
    os.environ['ARC_COMP_DIR'] = os.path.dirname(_comp[0])
print('ARC_COMP_DIR', os.getenv('ARC_COMP_DIR'))
os.environ['ARC_EFFORT'] = 'xhigh'          # ARC Prize 27B semi-private: low 22.8%, xhigh 42.4%
os.environ['ARC_THINK_BUDGET'] = '32000'    # both attempts
os.environ['ARC_SINGLE_REQUEST'] = '1'   # think + close + answer in one request; re-prefill only capped attempts
os.environ['ARC_SINGLE_SLACK'] = '256'
os.environ['ARC_ENGINE_FORCED_CLOSE'] = '1'   # vLLM thinking_token_budget + reasoning_end_str
os.environ['ARC_PREVIEW_SECONDS'] = '2700'
os.environ['ARC_THINK_BUDGET'] = '56000'
os.environ['ARC_THINK_BUDGET2'] = '32000'
os.environ['ARC_PROGRAMS'] = '1'
os.environ['ARC_EXTRA_ATTEMPTS'] = '1'
os.environ['ARC_REPAIR_THINK'] = '24000,16000'
os.environ['ARC_DEADLINE_CAP'] = '1'
os.environ['ARC_SPEC_EXTRA'] = '{"use_local_argmax_reduction":true}'
os.environ['ARC_AUG2'] = 'shuffle'
os.environ['ARC_TEMPERATURE0'] = '0.6'  # attempt_1: thinking and grid
os.environ['ARC_TEMPERATURE1'] = '0.6'  # attempt_2: thinking and grid
os.environ['ARC_PREVIEW_TASKS'] = 'e8686506'  # commit/preview only; the rerun solves every task
# FP8-KV startup failures retry these '||'-separated configs in turn
os.environ.setdefault('ARC_FALLBACK_SERVER_ARGS', '--max-num-batched-tokens 4096 || --max-num-batched-tokens 2048 --enforce-eager')  # BF16 KV, then eager
os.environ['ARC_FALLBACK_ANY'] = '1'   # any FP8-KV startup failure falls back to BF16 KV
os.environ.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', VLLM_NO_USAGE_STATS='1',
                  DO_NOT_TRACK='1', VLLM_USE_FLASHINFER_SAMPLER='0', TOKENIZERS_PARALLELISM='false')
print(subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout)


ARC_COMP_DIR /kaggle/input/competitions/arc-prize-2026-arc-agi-2
0, NVIDIA L4, 23034 MiB
1, NVIDIA L4, 23034 MiB
2, NVIDIA L4, 23034 MiB
3, NVIDIA L4, 23034 MiB



In [2]:
# Offline vLLM 0.30.0: install from whichever attached wheelhouse matches this image's Python
# (thangnm1/arc-vllm-wheels = cp312, thangnm1/arc-vllm-wheels-py313 = cp313; both pip-downloaded on Kaggle).
import glob, os, subprocess, sys, time
from pathlib import Path
t = time.time()
tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
try:
    import vllm
    have = vllm.__version__
except Exception:
    have = None
if have != "0.30.0":
    dirs = sorted({os.path.dirname(w) for w in glob.glob('/kaggle/input/**/vllm-0.30.0-*.whl', recursive=True)
                   if glob.glob(os.path.join(os.path.dirname(w), f'torch-2.13*-{tag}-*.whl'))})
    if not dirs:
        raise RuntimeError(f'no attached vLLM 0.30.0 wheelhouse with a {tag} torch wheel')
    print('wheelhouses', dirs, flush=True)
    links = [x for d in dirs for x in ('--find-links', d)]
    # keep the image's transformers when vLLM accepts it (5.16.1 on the 2026-10 image)
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', *links, 'vllm==0.30.0'],
                       capture_output=True, text=True, timeout=1800)
    print(r.stdout[-3000:], r.stderr[-3000:], flush=True)
    r.check_returncode()
    # The image's torchaudio 2.11.0+cu128 satisfies pip's version check but not torch 2.13+cu130, and
    # transformers imports it whenever it is installed; text-only serving does not need it.
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'], capture_output=True, text=True)
# Import the whole server chain now so an environment break fails here, not after the model load starts.
r = subprocess.run([sys.executable, '-c', 'import torch, vllm, transformers; import vllm.entrypoints.openai.api_server; '
                    'print("torch", torch.__version__, torch.version.cuda, "gpus", torch.cuda.device_count(), '
                    '"vllm", vllm.__version__, "transformers", transformers.__version__)'],
                   capture_output=True, text=True, timeout=900)
print(r.stdout, r.stderr[-4000:] if r.returncode else '', flush=True)
r.check_returncode()
# Triton links generated kernels against libcuda.so; Kaggle ships only libcuda.so.1.
libs = sorted(glob.glob('/usr/lib/x86_64-linux-gnu/libcuda.so.1') + glob.glob('/usr/local/nvidia/lib64/libcuda.so.1'))
Path('/tmp/cudalib').mkdir(exist_ok=True)
if libs and not Path('/tmp/cudalib/libcuda.so').exists():
    Path('/tmp/cudalib/libcuda.so').symlink_to(libs[0])
os.environ['LIBRARY_PATH'] = '/tmp/cudalib:' + os.environ.get('LIBRARY_PATH', '')
print('install cell', round(time.time() - t), 's')


wheelhouses ['/kaggle/input/notebooks/thangnm1/arc-vllm-wheels-py313/wheels']
hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: fastapi
    Found existing installation: fastapi 0.141.1
    Uninstalling fastapi-0.141.1:
      Successfully uninstalled fastapi-0.141.1
  Attempting uninstall: torch
    Found existing installation: torch 2.11.0+cu128
    Uninstalling torch-2.11.0+cu128:
      Successfully uninstalled torch-2.11.0+cu128
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.26.0+cu128
    Uninstalling torchvision-0.26.0+cu128:
      Successfully uninstalled torchvision-0.26.0+cu128
  dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have prot

In [3]:
%%writefile patch_vllm_thinking_budget.py
"""Patch vLLM 0.30's Model Runner V2 thinking-budget sampler in place (idempotent).

1. vllm#58231 insurance: the forced end-string token gets logit 1e4 instead of 1e9 (1e9 can lose the split top-p
   pivot in fp32; harmless for top_k=20 + top_p=0.95, but cheap to rule out).
2. vllm#54906 guard: the reasoning marker-token tensors (start / natural end / forced end) are small device tensors
   created before startup profiling; on Qwen3.8 + MTP they can come out zeroed, which silently disables the budget.
   Keep pinned CPU copies and refresh the device tensors before every use (three tiny H2D copies per sampled step).

usage: python patch_vllm_thinking_budget.py   (patches the vllm importable by this interpreter)
"""
import os
import sys

import vllm

PATH = os.path.join(os.path.dirname(vllm.__file__), "v1", "worker", "gpu", "sample", "thinking_budget.py")
MARK = "# arc-patch: thinking budget guard v1"


def main():
    src = open(PATH).read()
    if MARK in src:
        print("ALREADY_PATCHED", PATH)
        return 0
    if vllm.__version__ != "0.30.0":
        print(f"WARNING: written against vllm 0.30.0, found {vllm.__version__}", file=sys.stderr)

    old_force = "tl.store(logits_ptr + token_idx * logits_stride + force_token_id, 1.0e9)"
    assert src.count(old_force) == 1, "forced-logit store not found"
    src = src.replace(old_force, "tl.store(logits_ptr + token_idx * logits_stride + force_token_id, 1.0e4)")

    old_init = """        self.reasoning_end_token_ids = torch.tensor(
            end_ids, dtype=torch.int32, device=self.device
        )
"""
    new_init = old_init + f"""        {MARK}: pinned CPU masters of the marker ids (vllm#54906)
        self._marker_cpu = [
            torch.tensor(ids, dtype=torch.int32).pin_memory()
            for ids in (start_ids, natural_end_ids, end_ids)
        ]
"""
    assert src.count(old_init) == 1, "marker tensor init not found"
    src = src.replace(old_init, new_init)

    old_apply = """        apply_thinking_budget(
            logits,
            idx_mapping,"""
    new_apply = """        for dst, cpu in zip(
            (
                self.reasoning_start_token_ids,
                self.natural_reasoning_end_token_ids,
                self.reasoning_end_token_ids,
            ),
            self._marker_cpu,
        ):
            dst.copy_(cpu, non_blocking=True)
        apply_thinking_budget(
            logits,
            idx_mapping,"""
    assert src.count(old_apply) == 1, "apply call not found"
    src = src.replace(old_apply, new_apply)

    open(PATH + ".orig", "w").write(open(PATH).read())
    open(PATH, "w").write(src)
    print("PATCHED", PATH)
    return 0


if __name__ == "__main__":
    sys.exit(main())


Writing patch_vllm_thinking_budget.py


In [4]:
import subprocess, sys
r = subprocess.run([sys.executable, 'patch_vllm_thinking_budget.py'], capture_output=True, text=True)
print(r.stdout, r.stderr)
assert 'PATCHED' in r.stdout, 'vLLM thinking-budget patch failed'


PATCHED /usr/local/lib/python3.13/dist-packages/vllm/v1/worker/gpu/sample/thinking_budget.py
 


In [5]:
# Qwen3.8-27B RFT v1 as NVFP4 (unsloth recipe, requantized 2026-10-14: GPTQ NVFP4 MLP layers 0-55, served on L4 as
# Marlin W4A16; per-channel FP8 W8A8 attention / GDN / lm_head / MLP 56-63; BF16 MTP head; static FP8 KV scales).
# The checkpoint ships with the EOS rows of its FP8 lm_head (weight + weight_scale) zeroed (EOSFIX.json), so it is
# served in place from the read-only dataset: no overlay copy. Public eval paired with BF16 (172 inputs, same seeds,
# Marlin kernels forced): grid 40 vs 41, grid or train-perfect program 51 vs 50.
import json, os, time
from pathlib import Path
t = time.time()
cands = [c.parent for c in Path('/kaggle/input').rglob('config.json')
         if '"qwen3_5' in c.read_text() and 'compressed-tensors' in c.read_text()
         and (c.parent / 'tokenizer_config.json').exists()]
assert len(cands) == 1, cands
main = cands[0]
assert (main / 'EOSFIX.json').exists(), 'the NVFP4 checkpoint must ship EOS-patched (EOSFIX.json)'
wm = json.loads((main / 'model.safetensors.index.json').read_text())['weight_map']
assert any(k.startswith('mtp.') for k in wm), 'checkpoint has no MTP head'
os.environ['ARC_MODEL_DIR'] = str(main)
os.environ.pop('ARC_MTP_MODEL', None)                # MTP head comes from the checkpoint itself
print('nvfp4 model', main, json.loads((main / 'EOSFIX.json').read_text())['eos_rows_zeroed'], f'| {time.time() - t:.0f}s')


nvfp4 model /kaggle/input/datasets/thangnm1/qwen38-27b-arc-rft-v1-nvfp4 [248044, 248046] | 0s


In [6]:
%%writefile arc_kaggle_runner.py
"""Qwen3.8-27B ARC-AGI-2 runner for Kaggle (offline vLLM server, thinking low).

Setup validated on an RTX 5090 (arc_research_2026/qwen38_5090_20261003):
  * ARC Prize harness prompt, reasoning_effort=low, 2 attempts per test input;
  * every attempt is two /v1/completions requests on token ids: phase T thinks until
    </think> with <|im_end|>/<|endoftext|> banned (Qwen3.8 otherwise ends its turn
    mid-thought on grid enumerations, vLLM #55420), phase F answers under a JSON-grid
    grammar so EOS cannot cut a grid;
  * a complete submission.json (same schema as the 33.06 notebook) is written before
    the model loads and atomically after every finished attempt, so it can be
    submitted at any moment; unfinished inputs keep [[0]] placeholders.
Order: tasks with the shortest prompts first; first attempts are queued before
second attempts. The optional strict pass barrier waits for all first attempts
to finish before starting any second attempts.
Preview (no KAGGLE_IS_COMPETITION_RERUN): public evaluation set, scored live from
arc-agi_evaluation_solutions.json (read only by the scoring helper). Rerun: hidden test
set, no solutions anywhere.
"""
from __future__ import annotations

import argparse
import asyncio
import hashlib
import json
import os
import re
import signal
import socket
import subprocess
import sys
import time
from pathlib import Path

import httpx

COMP = Path(os.getenv("ARC_COMP_DIR", "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"))
THINK_END = 248069                  # '</think>' (not a special token)
EOS_IDS = (248046, 248044)          # <|im_end|>, <|endoftext|>
CONTEXT_MARGIN = 16
SINGLE_REQUEST = os.getenv("ARC_SINGLE_REQUEST", "0") == "1"
SINGLE_SLACK = int(os.getenv("ARC_SINGLE_SLACK", "3000"))   # tokens past the think cap a single request may run
ENGINE_FORCED_CLOSE = os.getenv("ARC_ENGINE_FORCED_CLOSE", "0") == "1"   # close capped thinking inside the request
MIN_THINK_TOKENS = 32
FORCED_CLOSE = ("\n\nConsidering the limited time by the user, I have to give the solution "
                "based on the thinking directly now.\n")
# Compact JSON grid; used instead of GRID_SCHEMA on EOS-patched checkpoints: once the last bracket closes only
# EOS is allowed, which the grammar forces even though the patched lm_head never samples EOS on its own.
GRID_REGEX = r"\[\[[0-9](, ?[0-9]){0,29}\](, ?\[[0-9](, ?[0-9]){0,29}\]){0,29}\]"  # <= 30 x 30
GRID_SCHEMA = {"type": "array", "minItems": 1, "maxItems": 30,
               "items": {"type": "array", "minItems": 1, "maxItems": 30,
                         "items": {"type": "integer", "minimum": 0, "maximum": 9}}}
HARNESS_PROMPT = """You are participating in a puzzle solving competition. You are an expert at solving puzzles.

Below is a list of input and output pairs with a pattern. Your goal is to identify the pattern or transformation in the training examples that maps the input to the output, then apply that pattern to the test input to give a final output.

Respond in the format of the training output examples

--Training Examples--
{training_examples}
--End of Training Examples--

--Test Input--
{test_input}
--End of Test Input--

Your response:
"""


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


def atomic_json(path, value):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(value))
    tmp.replace(path)


def valid_grid(g) -> bool:
    return (isinstance(g, list) and 1 <= len(g) <= 30 and isinstance(g[0], list)
            and 1 <= len(g[0]) <= 30 and all(isinstance(r, list) and len(r) == len(g[0])
            and all(type(v) is int and 0 <= v <= 9 for v in r) for r in g))


def last_grid(text):
    dec = json.JSONDecoder()
    found = None
    for m in re.finditer(r"\[\s*\[", text or ""):
        try:
            cand, _ = dec.raw_decode(text[m.start():])
        except json.JSONDecodeError:
            continue
        if valid_grid(cand):
            found = cand
    return found


def recover_unclosed_outer_grid(text):
    """Recover only a complete JSON grid missing its final outer bracket.

    A constrained decoder can emit whitespace indefinitely after the last
    row while waiting to close the outer array. Never invent row values or
    repair a partly generated row.
    """
    stripped = (text or "").strip()
    if not stripped.startswith("[[") or not stripped.endswith("]"):
        return None
    try:
        grid = json.loads(stripped + "]")
    except json.JSONDecodeError:
        return None
    return grid if valid_grid(grid) else None


# Geometric views (forward, inverse). Whether a transposed second attempt helps
# RadixArk must be measured separately; existing view comparisons used other checkpoints.
AUGS = {"none": (lambda g: g, lambda g: g),
        "transpose": (lambda g: [list(r) for r in zip(*g)], lambda g: [list(r) for r in zip(*g)]),
        "rot90": (lambda g: [list(r) for r in zip(*g[::-1])], lambda g: [list(r) for r in zip(*g)][::-1])}


AUGS["shuffle"] = (lambda g: g, lambda g: g)


def view_of(task, aug, tid):
    """The prompt view of a task for one attempt: geometric augmentation, or a reordered list of training examples."""
    if aug != "shuffle":
        return augment_task(task, AUGS[aug][0])
    import random as _random
    idx = list(range(len(task["train"])))
    if len(idx) > 1:
        rng = _random.Random(seed_of(tid, "demo_order"))
        while idx == sorted(idx):
            rng.shuffle(idx)
    return {"train": [task["train"][i] for i in idx], "test": [{"input": t["input"]} for t in task["test"]]}


def augment_task(task, fwd):
    return {"train": [{"input": fwd(p["input"]), "output": fwd(p["output"])} for p in task["train"]],
            "test": [{"input": fwd(t["input"])} for t in task["test"]]}


def harness_prompt(task, k):
    ex = ""
    for i, p in enumerate(task["train"]):
        ex += (f"--Example {i}-- \n\n INPUT: \n\n" + json.dumps(p["input"]) + "\n\n"
               + "OUTPUT: \n\n" + json.dumps(p["output"]) + "\n\n")
    return HARNESS_PROMPT.format(training_examples=ex, test_input=json.dumps(task["test"][k]["input"]))


def seed_of(*parts):
    return int(hashlib.sha256("/".join(map(str, parts)).encode()).hexdigest()[:8], 16)


# ----------------------------------------------------------------- submission / scoring
def build_submission(tasks, answers, programs=None):
    sub = {}
    for tid, task in tasks.items():
        rows = []
        for k in range(len(task["test"])):
            # 1) any candidate two or more attempts agree on (programs or grids, most votes first); 2) train-perfect
            # program outputs, then the direct grids, each pool by votes (ties keep attempt order). Policy simulation
            # on the public eval with 3 attempts: 40.79 vs 40.51 programs-then-grids vs 39.95 program+grid.
            progs = [g for g in (programs or {}).get(f"{tid}/{k}", []) if g is not None]
            grids = [g for g in answers.get(f"{tid}/{k}", []) if g is not None]
            votes = {}
            for g in progs + grids:
                votes[json.dumps(g)] = votes.get(json.dumps(g), 0) + 1
            got = []
            for g in sorted(progs + grids, key=lambda g: -votes[json.dumps(g)]):
                if votes[json.dumps(g)] >= 2 and g not in got:
                    got.append(g)
            for pool in (progs, grids):
                for g in sorted(pool, key=lambda g: -votes[json.dumps(g)]):
                    if g not in got:
                        got.append(g)
            a1 = got[0] if got else [[0]]
            a2 = got[1] if len(got) > 1 else [[0]]   # a copy of attempt_1 can never add a solve
            rows.append({"attempt_1": a1, "attempt_2": a2})
        sub[tid] = rows
    return sub


def score_submission(sub, solutions, task_ids):
    """Official ARC-AGI-2 Kaggle scoring unit: each test input has equal weight."""
    per = {}
    for tid in task_ids:
        gold = solutions[tid]
        ok = [any(sub[tid][k][a] == gold[k] for a in ("attempt_1", "attempt_2")) for k in range(len(gold))]
        per[tid] = ok
    return per


def preview_score_report(sub, tasks, solutions, selected_task_ids, answers):
    """Keep all/selected/started *input* denominators distinct for Kaggle."""
    per = score_submission(sub, solutions, list(tasks))
    selected = [ok for tid in selected_task_ids for ok in per[tid]]
    started_pairs = [(tid, k) for tid in selected_task_ids
                     for k in range(len(tasks[tid]["test"])) if f"{tid}/{k}" in answers]
    started_tasks = {tid for tid, _ in started_pairs}
    all_inputs = [ok for row in per.values() for ok in row]
    return {
        "score_all_inputs": round(100 * sum(all_inputs) / max(1, len(all_inputs)), 2),
        "score_selected_inputs": round(100 * sum(selected) / max(1, len(selected)), 2),
        "score_started_inputs": round(100 * sum(per[tid][k] for tid, k in started_pairs)
                                      / max(1, len(started_pairs)), 2),
        "selected_tasks": len(selected_task_ids),
        "selected_inputs": len(selected),
        "tasks_started": len(started_tasks),
        "inputs_started": len(started_pairs),
        "tasks_fully_solved": sum(1 for tid in selected_task_ids if all(per[tid])),
    }


# ----------------------------------------------------------------- vLLM server
def find_model_dir():
    if os.getenv("ARC_MODEL_DIR"):
        return os.environ["ARC_MODEL_DIR"]
    hits = []
    for cfg in Path("/kaggle/input").rglob("config.json"):
        try:
            c = json.loads(cfg.read_text())
        except (OSError, ValueError):
            continue
        # qwen3_5 = Qwen3.8-27B, qwen4_exp = Qwen3.8-Flash-Next (MoE); vision tower skipped by --language-model-only
        if c.get("model_type") in ("qwen3_5", "qwen4_exp") and (cfg.parent / "tokenizer_config.json").exists():
            hits.append(str(cfg.parent))
    if len(hits) != 1:
        raise RuntimeError(f"expected one Qwen3.8 model dir, found {hits}; set ARC_MODEL_DIR")
    return hits[0]


def start_server(a, model_dir):
    # A stale vLLM process from an earlier notebook cell must never be mistaken
    # for the server launched below. Otherwise /v1/models can answer from the
    # wrong process while this process is still loading (or about to fail).
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
        try:
            probe.bind(("127.0.0.1", a.port))
        except OSError as exc:
            raise RuntimeError(f"vLLM port {a.port} is already in use") from exc
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", model_dir,
           "--served-model-name", "qwen38", "--host", "127.0.0.1", "--port", str(a.port),
           "--tensor-parallel-size", str(a.tp), "--language-model-only", "--reasoning-parser", "qwen3",
           "--max-model-len", str(a.max_model_len), "--gpu-memory-utilization", str(a.gpu_util),
           "--max-num-seqs", str(a.max_num_seqs), "--seed", "0"] + a.extra_server_args.split()
    if getattr(a, "patched", False) and getattr(a, "mtp", 0) > 0:
        # vLLM 0.30 rejects logit_bias with speculative decoding, hence MTP only on EOS-patched checkpoints;
        # MTP + prefix caching is avoided (vLLM #53912)
        spec = {"method": "mtp", "num_speculative_tokens": a.mtp}
        spec.update(json.loads(os.getenv("ARC_SPEC_EXTRA", "{}")))   # extra speculative-config keys
        if os.getenv("ARC_MTP_MODEL"):            # separate draft checkpoint (e.g. albucino's int4 MTP for Flash-Next)
            spec["model"] = os.environ["ARC_MTP_MODEL"]
        cmd += ["--no-enable-prefix-caching", "--speculative-config", json.dumps(spec, separators=(",", ":"))]
    if ENGINE_FORCED_CLOSE:   # the forced end string = the client-side FORCED_CLOSE + </think>
        cmd += ["--reasoning-config", json.dumps({"reasoning_end_str": FORCED_CLOSE + "</think>"})]
    log("SERVER", " ".join(cmd))
    logf = open(a.server_log, "w")
    return subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True)


ERR_RE = re.compile(r"Error|Exception|out of memory|Killed|died|exitcode|signal \d+|Traceback")
MEM_RE = re.compile(r"out of memory|OutOfMemory|No available memory for the cache blocks|available KV cache memory|"
                    r"Free memory on device|cudaErrorMemoryAllocation|CUBLAS_STATUS_ALLOC_FAILED")
PROGRESS_RE = re.compile(r"Initialized PLE embedding|Loading weights took|Model loading took|GPU KV cache size|"
                         r"Available KV cache memory|CUDA graph|Graph capturing finished")


def root_cause(text, limit=30):
    """Error lines of the engine/worker processes (the APIServer traceback only says 'see root cause above')."""
    keep, seen = [], set()
    for line in text.splitlines():
        if "APIServer pid" in line or not (ERR_RE.search(line) or PROGRESS_RE.search(line)):
            continue
        body = re.sub(r"^\(\w+(?:_\w+)* pid=\d+\)\s*", "", line).strip()
        if body and body not in seen:
            seen.add(body)
            keep.append(line.strip()[:400])
    if len(keep) > limit:
        keep = keep[:limit // 3] + ["..."] + keep[-(limit - limit // 3):]
    return keep


def wait_server(proc, a, deadline):
    while time.monotonic() < deadline:
        if proc.poll() is not None:
            text = Path(a.server_log).read_text(errors="replace")
            raise RuntimeError("vLLM server exited during startup; root cause lines:\n" + "\n".join(root_cause(text))
                               + "\n--- log tail ---\n" + text[-1500:])
        try:
            if httpx.get(f"http://127.0.0.1:{a.port}/v1/models", timeout=5).status_code == 200:
                return
        except httpx.HTTPError:
            pass
        time.sleep(10)
    raise TimeoutError("vLLM server did not become ready in time")


def kv_tokens(a):
    m = re.findall(r"GPU KV cache size: ([\d,]+) tokens", Path(a.server_log).read_text())
    return int(m[-1].replace(",", "")) if m else None


def think_budget_of(a, att):
    """attempt_2 may run with its own cap (--think-budget2), e.g. a quick 24K first pass, then 40K."""
    return a.think_budget2 if att >= 1 and getattr(a, "think_budget2", 0) else a.think_budget   # extras use the attempt_2 cap


def full_request_kv_bound(prompt_tokens, think_budget, answer_budget, forced_suffix_tokens,
                          max_model_len):
    """Maximum resident tokens for the actual two-phase request, or None if skipped."""
    available_think = (max_model_len - prompt_tokens - forced_suffix_tokens
                       - answer_budget - CONTEXT_MARGIN)
    if available_think < MIN_THINK_TOKENS:
        return None
    return (prompt_tokens + min(think_budget, available_think) + forced_suffix_tokens
            + answer_budget + CONTEXT_MARGIN)


def max_selected_request_tokens(tok, tasks, order, a):
    """Tokenize the selected challenge-visible prompts, including the second view."""
    forced_suffix_tokens = (len(tok.encode(FORCED_CLOSE, add_special_tokens=False)) + 1
                            + len(tok.encode("\n\n", add_special_tokens=False)))
    bounds = []
    skipped = 0
    for att in range(a.attempts):
        fwd, _ = AUGS[a.aug2 if att == 1 else "none"]
        for tid in order:
            task = augment_task(tasks[tid], fwd)
            for k in range(len(task["test"])):
                prompt = harness_prompt(task, k)
                encoded = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                                  tokenize=True, add_generation_prompt=True,
                                                  return_dict=True, enable_thinking=True,
                                                  reasoning_effort=a.effort)
                bound = full_request_kv_bound(len(encoded["input_ids"]), think_budget_of(a, att),
                                              a.answer_budget, forced_suffix_tokens, a.max_model_len)
                if bound is None:
                    skipped += 1
                else:
                    bounds.append(bound)
    return (max(bounds) if bounds else None), skipped


def kv_safe_concurrency(kv_capacity, max_request_tokens, max_num_seqs, headroom_fraction):
    """A conservative client in-flight cap; this cannot prevent non-KV OOM."""
    if kv_capacity is None or kv_capacity <= 0:
        raise ValueError("KV capacity unavailable; cannot use kv-safe concurrency")
    if max_num_seqs < 1 or not 0 <= headroom_fraction < 1:
        raise ValueError("invalid max_num_seqs or KV headroom fraction")
    if max_request_tokens is None:
        return 1, int(kv_capacity * (1 - headroom_fraction))
    if max_request_tokens <= 0:
        raise ValueError("max_request_tokens must be positive")
    usable = int(kv_capacity * (1 - headroom_fraction))
    return max(1, min(max_num_seqs, usable // max_request_tokens)), usable


def temperature_of(a, attempt):
    if attempt < 0:
        raise ValueError("attempt index must be non-negative")
    return a.temperature0 if attempt == 0 else a.temperature1   # extra attempts sample like attempt_2


# ----------------------------------------------------------------- generation
class Gen:
    def __init__(self, a, model_dir, concurrency):
        from transformers import AutoTokenizer
        self.a = a
        self.tok = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
        assert self.tok.convert_tokens_to_ids("</think>") == THINK_END
        self.close_ids = self.tok.encode("\n\n", add_special_tokens=False)
        self.forced_ids = self.tok.encode(FORCED_CLOSE, add_special_tokens=False) + [THINK_END] + self.close_ids
        self.http = httpx.AsyncClient(timeout=httpx.Timeout(7200, connect=30),
                                      limits=httpx.Limits(max_connections=concurrency + 8))
        self.url = f"http://127.0.0.1:{a.port}/v1/completions"

    def render(self, text):
        enc = self.tok.apply_chat_template([{"role": "user", "content": text}], tokenize=True,
                                           add_generation_prompt=True, return_dict=True,
                                           enable_thinking=True, reasoning_effort=self.a.effort)
        ids = list(enc["input_ids"])
        if self.a.effort in ("low", "xhigh") and f"Reasoning effort is set to {self.a.effort}" not in self.tok.decode(ids[:64]):
            raise ValueError(f"chat template did not render reasoning_effort={self.a.effort}")
        return ids

    async def complete(self, ids, max_tokens, seed, stop=None, mask_eos=False, schema=None, regex=None,
                       temperature=1.0, thinking_budget=None):
        a = self.a
        max_tokens = min(max_tokens, a.max_model_len - len(ids) - 16)
        if max_tokens < 32:
            return {"text": "", "token_ids": [], "finish_reason": "context_full"}
        body = {"model": "qwen38", "prompt": ids, "max_tokens": max_tokens, "temperature": temperature,
                "top_p": 0.95, "top_k": 20, "min_p": 0.0, "presence_penalty": 0.0,
                "repetition_penalty": 1.0, "seed": seed, "skip_special_tokens": False,
                "return_token_ids": True}
        if stop:
            body["stop_token_ids"] = stop
        if mask_eos:
            body["logit_bias"] = {str(t): -100 for t in EOS_IDS}
        if schema is not None:
            body["structured_outputs"] = {"json": schema}
        if regex is not None:
            body["structured_outputs"] = {"regex": regex}
        if thinking_budget is not None:
            body["thinking_token_budget"] = thinking_budget
        log("REQUEST_SAMPLER", "phase", "thinking" if stop else "grid", "seed", seed,
            "temperature", body["temperature"], "top_p", body["top_p"], "top_k", body["top_k"],
            "min_p", body["min_p"], "presence_penalty", body["presence_penalty"],
            "repetition_penalty", body["repetition_penalty"], "max_tokens", body["max_tokens"])
        for _ in range(3):
            try:
                r = await self.http.post(self.url, json=body)
                if 400 <= r.status_code < 500:
                    return {"text": "", "token_ids": [], "finish_reason": "error", "error": r.text[:300]}
                r.raise_for_status()
                return r.json()["choices"][0]
            except Exception as exc:  # transport errors, server restarts: retry, never lose the attempt task
                err = repr(exc)[:300]
                await asyncio.sleep(20)
        return {"text": "", "token_ids": [], "finish_reason": "error", "error": err}

    async def repair(self, prompt, program, feedback, seed, think_cap, temperature):
        """Repair turn: harness prompt, the earlier program as the assistant reply, sandbox feedback; the model thinks
        (engine-capped) and a grammar forces the corrected transform (Colab test: qwen38_repair_20261014)."""
        msgs = [{"role": "user", "content": prompt},
                {"role": "assistant", "content": "```python\n" + program.strip() + "\n```"},
                {"role": "user", "content": REPAIR_ASK.format(feedback=feedback)}]
        ids = list(self.tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_dict=True,
                                                enable_thinking=True, reasoning_effort=self.a.effort)["input_ids"])
        core = self.forced_ids[:len(self.forced_ids) - len(self.close_ids)]
        t0 = time.monotonic()
        ch = await self.complete(ids, think_cap + len(core) + PROGRAM_BUDGET, seed,
                                 regex=r"\n{0,2}" + regex_literal(REPAIR_CUE) + r"[^`]+[`][`][`]",
                                 temperature=temperature, thinking_budget=think_cap + 1)
        toks = list(ch.get("token_ids") or [])
        res = {"grid": None, "closed": False, "think_tokens": 0, "answer_tokens": 0, "prompt_tokens": len(ids),
               "generated_tokens": len(toks), "error": ch.get("error"), "seconds": round(time.monotonic() - t0, 1),
               "mode": "repair", "program_code": None}
        if THINK_END in toks:
            j = toks.index(THINK_END)
            text = self.tok.decode(toks[j + 1:], skip_special_tokens=False)
            body = text.split("def transform(grid):\n", 1)[-1].split("```", 1)[0]
            res.update(closed=True, think_tokens=j + 1,
                       program_code="def transform(grid):\n" + body if body.strip() else None)
        return res

    async def attempt(self, prompt, seed, think_budget=None, temperature=1.0):
        a = self.a
        requested = think_budget or a.think_budget
        ids = self.render(prompt)
        t0 = time.monotonic()
        # Reserve the entire answer budget even if thinking reaches its cap and
        # needs the longer forced-close suffix. complete() otherwise clips the
        # answer phase silently when a long prompt consumes the context window.
        available_think = (a.max_model_len - len(ids) - len(self.forced_ids)
                           - a.answer_budget - CONTEXT_MARGIN)
        if available_think < MIN_THINK_TOKENS:
            log("SKIP_CONTEXT", "prompt", len(ids), "available_think", available_think,
                "minimum", MIN_THINK_TOKENS, "answer_budget", a.answer_budget)
            return {"grid": None, "closed": False, "think_tokens": 0, "answer_tokens": 0,
                    "prompt_tokens": len(ids), "answer_finish": "skipped", "think_finish": "skipped",
                    "error": "insufficient_context_for_thinking_and_full_answer_budget",
                    "seconds": round(time.monotonic() - t0, 1), "answer_tail": ""}
        think_cap = min(requested, available_think)
        if think_cap < requested:
            log("REDUCE_THINK_CAP", "prompt", len(ids), "requested", requested,
                "effective", think_cap, "answer_budget", a.answer_budget)
        if SINGLE_REQUEST and getattr(a, "patched", False):
            result = await self.attempt_single(ids, think_cap, seed, temperature, t0)
            if think_cap < requested:
                result["effective_think_budget"] = think_cap
            return result
        # EOS-patched checkpoints cannot sample <|im_end|>/<|endoftext|>; others get the per-request mask
        ch = await self.complete(ids, think_cap, seed, stop=[THINK_END], mask_eos=not getattr(a, "patched", False),
                                 temperature=temperature)
        think = list(ch.get("token_ids") or [])
        while think and think[-1] in EOS_IDS:
            think.pop()
        closed = bool(think) and think[-1] == THINK_END
        if not closed and ch.get("stop_reason") == THINK_END:
            think.append(THINK_END)
            closed = True
        prefix = ids + think + (self.close_ids if closed else self.forced_ids)
        if getattr(a, "patched", False):
            ch2 = await self.complete(prefix, a.answer_budget, seed + 1, regex=GRID_REGEX, temperature=temperature)
        else:
            ch2 = await self.complete(prefix, a.answer_budget, seed + 1, schema=GRID_SCHEMA, temperature=temperature)
        text = ch2.get("text") or ""
        grid = last_grid(text)
        recovered = False
        if grid is None and ch2.get("finish_reason") == "length":
            grid = recover_unclosed_outer_grid(text)
            recovered = grid is not None
        result = {"grid": grid, "recovered_grid": recovered,
                "closed": closed, "think_tokens": len(think),
                "answer_tokens": len(ch2.get("token_ids") or []), "prompt_tokens": len(ids),
                "answer_finish": ch2.get("finish_reason"), "think_finish": ch.get("finish_reason"),
                "error": ch.get("error") or ch2.get("error"), "seconds": round(time.monotonic() - t0, 1),
                "answer_tail": text[-300:]}
        if think_cap < requested:
            result["effective_think_budget"] = think_cap
        return result


    async def attempt_single(self, ids, think_cap, seed, temperature, t0):
        """Think, close and answer in one request. The grid regex is enforced only after </think> (server-side
        --reasoning-parser qwen3), so no second prefill of prompt + thinking is needed for closed attempts.
        No </think> within think_cap + SINGLE_SLACK tokens -> capped: the first think_cap tokens are force-closed and
        answered exactly as in the two-request path (same thinking budget, same forced suffix)."""
        a = self.a
        core = self.forced_ids[:len(self.forced_ids) - len(self.close_ids)]   # FORCED_CLOSE ids + </think>
        if ENGINE_FORCED_CLOSE:
            # the budget counts the prompt's trailing newline after <think>, hence + 1 = think_cap generated tokens
            ch = await self.complete(ids, think_cap + len(core) + a.answer_budget + (PROGRAM_BUDGET if PROGRAMS else 0), seed,
                                     regex=PROGRAM_REGEX if PROGRAMS else r"\n{0,2}" + GRID_REGEX,
                                     temperature=temperature, thinking_budget=think_cap + 1)
        else:
            ch = await self.complete(ids, think_cap + SINGLE_SLACK, seed, regex=r"\n{0,2}" + GRID_REGEX,
                                     temperature=temperature)
        toks = list(ch.get("token_ids") or [])
        err = ch.get("error")
        recovered = False
        program = None
        if THINK_END in toks:
            k = toks.index(THINK_END)
            think, ans = toks[:k + 1], toks[k + 1:]
            engine_forced = ENGINE_FORCED_CLOSE and k + 1 >= len(core) and toks[k + 1 - len(core):k + 1] == core
            closed = not engine_forced
            text = self.tok.decode(ans, skip_special_tokens=False)
            cut = text.find("\n\nThe transformation above") if PROGRAMS else -1
            if cut >= 0:   # grid first, then the program written in the same request
                body = text[cut:].split("def transform(grid):\n", 1)[-1].split("```", 1)[0]
                program = "def transform(grid):\n" + body if body.strip() else None
                text = text[:cut]
            grid = last_grid(text)
            finish = ch.get("finish_reason")
            mode = "engine_forced" if engine_forced else "single"
            if grid is None:   # answer cut by max_tokens (or malformed): answer request on the closed thinking
                ch2 = await self.complete(ids + think + self.close_ids, a.answer_budget, seed + 1, regex=GRID_REGEX,
                                          temperature=temperature)
                text = ch2.get("text") or ""
                ans = list(ch2.get("token_ids") or [])
                grid = last_grid(text)
                if grid is None and ch2.get("finish_reason") == "length":
                    grid = recover_unclosed_outer_grid(text)
                    recovered = grid is not None
                finish, err, mode = ch2.get("finish_reason"), err or ch2.get("error"), "single+answer"
        else:
            think, closed = toks[:think_cap], False
            ch2 = await self.complete(ids + think + self.forced_ids, a.answer_budget, seed + 1, regex=GRID_REGEX,
                                      temperature=temperature)
            text = ch2.get("text") or ""
            ans = list(ch2.get("token_ids") or [])
            grid = last_grid(text)
            if grid is None and ch2.get("finish_reason") == "length":
                grid = recover_unclosed_outer_grid(text)
                recovered = grid is not None
            finish, err, mode = ch2.get("finish_reason"), err or ch2.get("error"), "forced_fallback" if ENGINE_FORCED_CLOSE else "forced"
        think_tokens = len(think) - len(core) if mode.startswith("engine_forced") else len(think)
        return {"grid": grid, "recovered_grid": recovered, "closed": closed, "think_tokens": think_tokens,
                "answer_tokens": len(ans), "prompt_tokens": len(ids), "generated_tokens": len(toks),
                "answer_finish": finish, "think_finish": ch.get("finish_reason"), "mode": mode,
                "error": err, "seconds": round(time.monotonic() - t0, 1), "answer_tail": text[-300:],
                "program_code": program}


# ----------------------------------------------------------------- in-request program candidates (lever 7)
# After the grid the same request writes the rule as Python (grammar: grid, fixed cue, code body, closing fence); a
# program that reproduces every training pair supplies a candidate that goes first in the submission. Public eval
# (RFT v1, 32K, 172 inputs): 37 train-perfect, 27 right on test, 9 rescues, 0 losses; 31.53 vs 25.69 grid only, for
# ~1.5K extra tokens per attempt and no re-prefill.
PROGRAMS = os.getenv("ARC_PROGRAMS", "0") == "1"
PROGRAM_BUDGET = int(os.getenv("ARC_PROGRAM_BUDGET", "3000"))
EXTRA_ATTEMPTS = int(os.getenv("ARC_EXTRA_ATTEMPTS", "0"))   # further attempts for inputs still undecided after attempt_2
EXTRA_MIN_SECONDS = float(os.getenv("ARC_EXTRA_MIN_SECONDS", "2100"))   # an extra attempt (~32 min) must finish before the hard stop
# repair turns for inputs whose best program is partly right: think budget per sample, e.g. "24000,16000" = a 24K turn,
# then (only if it gave no train-perfect program) a 16K one. Colab, public eval, the 42 near/partial inputs, one sample
# each: 16K +7 / +2 (two seeds), 24K +7, 32K +7 newly solved inputs; 24K + 16K together +11.
REPAIR_THINKS = [int(x) for x in os.getenv("ARC_REPAIR_THINK", "0").split(",") if x.strip() and int(x) > 0]
REPAIR_THINK = REPAIR_THINKS[0] if REPAIR_THINKS else 0
REPAIR_SAMPLES = int(os.getenv("ARC_REPAIR_SAMPLES", str(max(1, len(REPAIR_THINKS)))))
REPAIR_MIN_SECONDS = float(os.getenv("ARC_REPAIR_MIN_SECONDS", "1800"))   # a 24K turn takes ~25 min at full concurrency
REPAIR_ASK = ("I ran your transform function on every training example:\n\n{feedback}\n\n"
              "Find what is wrong in the rule or in the code and write the corrected, complete transform function. It must "
              "reproduce every training output exactly and work on the test input. Pure Python only (no imports except "
              "copy/collections/itertools/math).")
REPAIR_CUE = "```python\ndef transform(grid):\n"
PROGRAM_SUFFIX = ("\n\nThe transformation above, written as a self-contained Python function (pure Python, no imports "
                  "except copy/collections/itertools/math) that maps any input grid of this task to its output grid:\n"
                  "```python\ndef transform(grid):\n")
PROGRAM_RUNNER = """
import json, sys
src, inputs = json.load(open(sys.argv[1]))
ns = {}
exec(compile(src, "transform", "exec"), ns)
out = []
for g in inputs:
    try:
        r = ns["transform"]([list(x) for x in g])
        out.append([[int(v) for v in row] for row in r])
    except Exception:
        out.append(None)
print(json.dumps(out))
"""


def regex_literal(s):
    """Literal for the structured-output regex: newlines as \\n, other punctuation as one-char classes."""
    out = []
    for ch in s:
        if ch == "\n":
            out.append(r"\n")
        elif ch.isalnum() or ch == " ":
            out.append(ch)
        elif ch in "]\\^":
            out.append("[\\" + ch + "]")
        else:
            out.append("[" + ch + "]")
    return "".join(out)


PROGRAM_REGEX = r"\n{0,2}" + GRID_REGEX + regex_literal(PROGRAM_SUFFIX) + r"[^`]+[`][`][`]"


def _program_limits():
    import resource
    resource.setrlimit(resource.RLIMIT_CPU, (10, 10))
    resource.setrlimit(resource.RLIMIT_AS, (1 << 30, 1 << 30))


def check_program(code, train_pairs, test_input):
    """Run transform() in a throwaway subprocess (10 s CPU, 1 GiB) on every training input and the test input."""
    import tempfile
    try:   # any failure of the temp dir, the subprocess or the JSON output = this candidate failed, the run goes on
        with tempfile.TemporaryDirectory() as d:
            Path(d, "in.json").write_text(json.dumps([code, [p["input"] for p in train_pairs] + [test_input]]))
            Path(d, "run.py").write_text(PROGRAM_RUNNER)
            r = subprocess.run([sys.executable, "-I", "run.py", "in.json"], cwd=d, capture_output=True, text=True,
                               timeout=20, preexec_fn=_program_limits)
            out = json.loads(r.stdout) if r.returncode == 0 else None
    except Exception:
        out = None
    if not isinstance(out, list) or len(out) != len(train_pairs) + 1:
        return {"status": "failed", "train_ok": 0, "train_n": len(train_pairs), "test_out": None}
    ok = sum(out[i] == p["output"] for i, p in enumerate(train_pairs))
    test = out[-1] if out[-1] is not None and valid_grid(out[-1]) else None
    return {"status": "ran", "train_ok": ok, "train_n": len(train_pairs), "test_out": test}


REPAIR_RUNNER = """
import json, sys, traceback
src, inputs = json.load(open(sys.argv[1]))
ns = {}
try:
    exec(compile(src, "transform", "exec"), ns)
except Exception as e:
    print(json.dumps({"compile_error": f"{type(e).__name__}: {e}"[:300]})); raise SystemExit
out = []
for g in inputs:
    try:
        r = ns["transform"]([list(x) for x in g])
        out.append({"grid": [[int(v) for v in row] for row in r]})
    except Exception as e:
        tb = traceback.extract_tb(e.__traceback__)
        line = tb[-1].lineno if tb else None
        out.append({"error": f"{type(e).__name__}: {e}"[:200] + (f" (line {line})" if line else "")})
print(json.dumps({"results": out}))
"""


def run_program_outputs(code, inputs):
    """transform() on each input in a throwaway subprocess (10 s CPU, 1 GiB): grids or per-input errors."""
    import tempfile
    with tempfile.TemporaryDirectory() as d:
        Path(d, "in.json").write_text(json.dumps([code, inputs]))
        Path(d, "run.py").write_text(REPAIR_RUNNER)
        try:
            r = subprocess.run([sys.executable, "-I", "run.py", "in.json"], cwd=d, capture_output=True, text=True,
                               timeout=20, preexec_fn=_program_limits)
            last = r.stdout.strip().splitlines()[-1] if r.stdout.strip() else ""
            return json.loads(last) if last else {"compile_error": (r.stderr or "no output")[-200:]}
        except subprocess.TimeoutExpired:
            return {"compile_error": "timeout (more than 10 s CPU)"}
        except Exception as exc:
            return {"compile_error": repr(exc)[:200]}


def repair_feedback(train, res):
    """Per training example: correct / error / size mismatch / differing cells (first 12); both grids for at most
    two wrong examples. Same text as the Colab repair test (qwen38_repair_20261014/prep_repair.py)."""
    if "compile_error" in res:
        return f"Your program could not run: {res['compile_error']}", 0
    lines, ok, shown = [], 0, 0
    for i, (p, r) in enumerate(zip(train, res.get("results", []))):
        exp = p["output"]
        if "error" in r:
            lines.append(f"Example {i}: ERROR while running: {r['error']}")
            continue
        got = r["grid"]
        if not valid_grid(got):
            lines.append(f"Example {i}: WRONG - the returned value is not a valid grid (rows of 1-30 ints 0-9).")
            continue
        if got == exp:
            ok += 1
            lines.append(f"Example {i}: correct.")
            continue
        if len(got) != len(exp) or len(got[0]) != len(exp[0]):
            lines.append(f"Example {i}: WRONG - expected a {len(exp)}x{len(exp[0])} grid, your program returned "
                         f"{len(got)}x{len(got[0])}.")
        else:
            diff = [(y, x, exp[y][x], got[y][x]) for y in range(len(exp)) for x in range(len(exp[0])) if exp[y][x] != got[y][x]]
            cells = ", ".join(f"({y},{x}) expected {e} got {g}" for y, x, e, g in diff[:12])
            lines.append(f"Example {i}: WRONG - {len(diff)} of {len(exp) * len(exp[0])} cells differ: {cells}"
                         + (" ..." if len(diff) > 12 else "") + ".")
        if shown < 2:
            shown += 1
            lines.append(f"  expected output:\n  {json.dumps(exp)}\n  your output:\n  {json.dumps(got)}")
    return "\n".join(lines), ok


# ----------------------------------------------------------------- main loop
async def solve(a, tasks, order, solutions, budget, model_dir, concurrency):
    gen = Gen(a, model_dir, concurrency)
    answers = {}
    programs = {}   # train-perfect program outputs per input
    inputs = [(tid, k) for tid in order for k in range(len(tasks[tid]["test"]))]
    pos = {key: i for i, key in enumerate(inputs)}
    # attempt_2 pass: inputs no. 1, 3, 5, ... (1-based, cost order) first, then 2, 4, 6, ...
    passes = [inputs[0::2] + inputs[1::2] if att == 1 else inputs for att in range(a.attempts)]
    work = [(tid, k, att) for att in range(a.attempts) for tid, k in passes[att]]
    done_att = {}
    checks = {}   # (attempt, aug, program_check, program_code) per input
    sem = asyncio.Semaphore(concurrency)
    lock = asyncio.Lock()
    req_log = open(a.out_dir / "requests.jsonl", "a")
    stats = {"attempts_done": 0, "attempt_0_done": 0, "attempt_0_grids": 0,
             "grids": 0, "closed": 0, "think_tokens": 0}

    def progress():
        sub = build_submission(tasks, answers, programs)
        atomic_json(a.out_dir / "submission.json", sub)
        rep = {"elapsed_s": round(time.monotonic() - budget["t0"], 1), "work_items": len(work), **stats,
               "inputs_with_answer": sum(1 for v in answers.values() if any(g is not None for g in v))}
        if solutions is not None:
            rep.update(preview_score_report(sub, tasks, solutions, order, answers))
        atomic_json(a.out_dir / "progress.json", rep)
        return rep

    def best_program(tid, k):
        """The partly right program to repair: un-augmented attempts only, most training pairs right (> 0)."""
        cands = [(pc.get("train_ok") or 0, -att, code) for att, aug, pc, code in checks.get((tid, k), [])
                 if code and aug == "none" and pc.get("status") == "ran"]
        if not cands:
            return None
        ok, _, code = max(cands)
        return code if ok > 0 else None

    async def repair_one(tid, k, j=0):
        async with sem:
            if time.monotonic() > budget["stop_new"] or budget["hard_stop"] - time.monotonic() < REPAIR_MIN_SECONDS:
                return
            if j > 0 and programs.get(f"{tid}/{k}"):
                return   # an earlier repair sample already produced a train-perfect program
            code = best_program(tid, k)
            task = tasks[tid]
            outs = await asyncio.to_thread(run_program_outputs, code, [p["input"] for p in task["train"]])
            fb, ok_before = repair_feedback(task["train"], outs)
            try:
                res = await gen.repair(harness_prompt(task, k), code, fb, seed_of(tid, k, f"repair{j}", a.seed_tag),
                                       REPAIR_THINKS[min(j, len(REPAIR_THINKS) - 1)], temperature_of(a, 1))
            except Exception as exc:
                res = {"grid": None, "closed": False, "think_tokens": 0, "mode": "repair", "program_code": None,
                       "error": f"{type(exc).__name__}: {exc}"[:300]}
            if res.get("program_code"):
                pc = await asyncio.to_thread(check_program, res["program_code"], task["train"], task["test"][k]["input"])
                res["program_check"] = pc
                if pc["status"] == "ran" and pc["train_n"] and pc["train_ok"] == pc["train_n"] and pc["test_out"] is not None:
                    res["program_grid"] = pc["test_out"] if valid_grid(pc["test_out"]) else None
        async with lock:
            if res.get("program_grid") is not None:
                programs.setdefault(f"{tid}/{k}", []).append(res["program_grid"])
                stats["repair_programs"] = stats.get("repair_programs", 0) + 1
            stats["repairs_done"] = stats.get("repairs_done", 0) + 1
            req_log.write(json.dumps({"task_id": tid, "test_index": k, "attempt": f"repair{j}",
                                      "train_ok_before": ok_before, **res}) + "\n")
            req_log.flush()
            progress()
            pc = res.get("program_check") or {}
            log("REPAIR", tid, k, j, f"train {ok_before} -> {pc.get('train_ok')}/{pc.get('train_n')}",
                "program" if res.get("program_grid") is not None else "no-program", f"{res.get('seconds')}s")

    async def one(tid, k, att):
        if isinstance(att, str) and att.startswith("repair"):
            return await repair_one(tid, k, int(att[6:]))
        async with sem:
            if time.monotonic() > budget["stop_new"]:
                return
            if att >= a.attempts and (budget["hard_stop"] - time.monotonic() < EXTRA_MIN_SECONDS
                                      or not wants_extra(f"{tid}/{k}")):
                return   # too late for a full extra attempt to finish, or a repair already produced a program
            aug = a.aug2 if att == 1 else "none"
            fwd, inv = AUGS[aug]
            started = time.monotonic()
            seed = seed_of(tid, k, att, a.seed_tag)
            sampler = {"temperature": temperature_of(a, att), "top_p": 0.95, "top_k": 20,
                       "min_p": 0.0, "presence_penalty": 0.0, "repetition_penalty": 1.0}
            log("ATTEMPT_SAMPLER", tid, k, att, "thinking_seed", seed, "grid_seed", seed + 1,
                json.dumps(sampler, sort_keys=True))
            cap = think_budget_of(a, att)
            if os.getenv("ARC_DEADLINE_CAP", "0") == "1":
                left = budget["hard_stop"] - time.monotonic() - 240
                fit = int(left * float(os.getenv("ARC_DEADLINE_RATE", "15"))) - 3000
                if fit < 4000:
                    log("DEADLINE_SKIP", tid, k, att, "fit", fit)
                    return
                if fit < cap:
                    log("DEADLINE_CAP", tid, k, att, cap, "->", fit)
                    cap = fit
            try:
                res = await gen.attempt(harness_prompt(view_of(tasks[tid], aug, tid), k),
                                        seed, cap, temperature=sampler["temperature"])
            except Exception as exc:
                # Individual request errors must not disappear inside an
                # asyncio.wait() task. Keep the submission valid and log the
                # failed attempt so its missing grid is distinguishable from
                # an attempt that was never scheduled.
                res = {"grid": None, "closed": False, "think_tokens": 0,
                       "answer_tokens": 0, "prompt_tokens": 0,
                       "answer_finish": "exception", "think_finish": "exception",
                       "error": f"{type(exc).__name__}: {exc}"[:300],
                       "seconds": round(time.monotonic() - started, 1), "answer_tail": ""}
            res["aug"] = aug
            res["seed"] = seed
            res["sampler"] = sampler
            if res["grid"] is not None:
                res["grid"] = inv(res["grid"])
                if not valid_grid(res["grid"]):
                    res["grid"] = None
            if res.get("program_code"):   # verify on the (same-view) training pairs; sandbox runs on the CPU
                view = view_of(tasks[tid], aug, tid)
                pc = await asyncio.to_thread(check_program, res["program_code"], view["train"], view["test"][k]["input"])
                res["program_check"] = pc
                if pc["status"] == "ran" and pc["train_n"] and pc["train_ok"] == pc["train_n"] and pc["test_out"] is not None:
                    pg = inv(pc["test_out"])
                    res["program_grid"] = pg if valid_grid(pg) else None
        async with lock:
            answers.setdefault(f"{tid}/{k}", [None] * (a.attempts + EXTRA_ATTEMPTS))[att] = res["grid"]
            if res.get("program_grid") is not None:
                programs.setdefault(f"{tid}/{k}", []).append(res["program_grid"])
                stats["program_grids"] = stats.get("program_grids", 0) + 1
            stats["attempts_done"] += 1
            stats["grids"] += res["grid"] is not None
            if att == 0:
                stats["attempt_0_done"] += 1
                stats["attempt_0_grids"] += res["grid"] is not None
            stats["closed"] += res["closed"]
            stats["think_tokens"] += res["think_tokens"]
            req_log.write(json.dumps({"task_id": tid, "test_index": k, "attempt": att, **res}) + "\n")
            req_log.flush()
            checks.setdefault((tid, k), []).append((att, res.get("aug"), res.get("program_check") or {},
                                                    res.get("program_code")))
            done_att.setdefault((tid, k), set()).add(att)
            if (EXTRA_ATTEMPTS and att < a.attempts and set(range(a.attempts)) <= done_att[(tid, k)]
                    and wants_extra(f"{tid}/{k}")):
                for extra in range(a.attempts, a.attempts + EXTRA_ATTEMPTS):
                    work.append((tid, k, extra))
                    enqueue(extra + 1, pos[(tid, k)], (tid, k, extra))
            if (REPAIR_THINK and att < a.attempts and set(range(a.attempts)) <= done_att[(tid, k)]
                    and not programs.get(f"{tid}/{k}") and best_program(tid, k) is not None):
                for j in range(REPAIR_SAMPLES):
                    work.append((tid, k, f"repair{j}"))
                    enqueue(a.attempts, j * len(inputs) + pos[(tid, k)], (tid, k, f"repair{j}"))   # all first samples first
            rep = progress()
            log("DONE", tid, k, att, "grid" if res["grid"] is not None else "nogrid",
                "think", res["think_tokens"], "closed" if res["closed"] else "forced", f"{res['seconds']}s",
                "| score_all_inputs", rep.get("score_all_inputs"),
                "score_selected_inputs", rep.get("score_selected_inputs"),
                "| attempts", rep["attempts_done"], "/", len(work))

    async def run_batch(batch):
        remaining = budget["hard_stop"] - time.monotonic()
        if not batch or remaining <= 0:
            return len(batch)
        pending = [asyncio.create_task(one(*w)) for w in batch]
        done, not_done = await asyncio.wait(pending, timeout=remaining,
                                            return_when=asyncio.FIRST_EXCEPTION)
        for task in not_done:
            task.cancel()
        if not_done:
            await asyncio.gather(*not_done, return_exceptions=True)
        # Fatal write/format errors must stop the run visibly; the notebook's
        # final validator can then retain an already written partial file.
        for task in done:
            task.result()
        return len(not_done)

    queue = asyncio.PriorityQueue()   # (level, position, seq): attempt_1, attempt_2, [repair], extras
    seq = [0]

    def enqueue(level, i, item):
        seq[0] += 1
        queue.put_nowait((level, i, seq[0], item))
    for att in range(a.attempts):
        for i, (tid, k) in enumerate(passes[att]):
            enqueue(att, i, (tid, k, att))

    def wants_extra(key):
        """No train-perfect program yet and the attempt grids do not already agree."""
        if programs.get(key):
            return False
        g = [x for x in answers.get(key, [])[:a.attempts] if x is not None]
        return not (len(g) == a.attempts and all(x == g[0] for x in g))

    async def run_queue():
        busy = 0

        async def worker():
            nonlocal busy
            while True:
                try:
                    _, _, _, item = queue.get_nowait()
                except asyncio.QueueEmpty:
                    if busy == 0:
                        return
                    await asyncio.sleep(1)
                    continue
                busy += 1
                try:
                    await one(*item)
                except asyncio.CancelledError:
                    raise
                except Exception as exc:   # one bad item must not cancel every other worker (FIRST_EXCEPTION)
                    log("ITEM_ERROR", item, f"{type(exc).__name__}: {exc}"[:300])
                finally:
                    busy -= 1

        workers = [asyncio.create_task(worker()) for _ in range(concurrency)]
        done, not_done = await asyncio.wait(workers, timeout=max(1.0, budget["hard_stop"] - time.monotonic()),
                                            return_when=asyncio.FIRST_EXCEPTION)
        for t in not_done:
            t.cancel()
        if not_done:
            await asyncio.gather(*not_done, return_exceptions=True)
        for t in done:
            t.result()
        return queue.qsize() + len(not_done)

    progress()
    cancelled = 0
    if a.strict_pass_order:
        for att in range(a.attempts):
            batch = [w for w in work if w[2] == att]
            n = await run_batch(batch)
            cancelled += n
            if n or time.monotonic() > budget["stop_new"]:
                cancelled += sum(1 for w in work if w[2] > att)
                break
    else:
        cancelled = await run_queue()
    rep = progress()
    log("SOLVE_END", json.dumps(rep), "cancelled_or_unstarted", cancelled)
    req_log.close()
    return rep


def make_budget(t0, total_seconds):
    """Leave time for the notebook watchdog, including on short previews.

    The notebook stops the runner at total_seconds - 240. submission.json is rewritten after every finished
    attempt, so long runs keep only 15/7 minute stop/hard margins (a 12 h rerun cuts in-flight attempts at
    11:53 and the watchdog stops at 11:56); short previews keep proportional margins.
    """
    if total_seconds <= 600:
        raise ValueError("run budget must exceed 600 seconds")
    hard_slack = min(420.0, max(300.0, total_seconds * 0.10))
    stop_slack = min(900.0, max(hard_slack + 120.0, total_seconds * 0.20))
    return {"t0": t0, "stop_new": t0 + total_seconds - stop_slack,
            "hard_stop": t0 + total_seconds - hard_slack}


def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--out-dir", default="/kaggle/working")
    p.add_argument("--tp", type=int, default=4)
    p.add_argument("--port", type=int, default=8000)
    p.add_argument("--max-model-len", type=int, default=81920)
    p.add_argument("--gpu-util", type=float, default=0.92)
    p.add_argument("--max-num-seqs", type=int, default=32)
    p.add_argument("--extra-server-args", default="")
    p.add_argument("--effort", default=os.getenv("ARC_EFFORT", "low"), choices=("low", "medium", "xhigh"),
                   help="reasoning_effort (or ARC_EFFORT); ARC Prize 27B semi-private: low 22.8%%, xhigh 42.4%%")
    p.add_argument("--attempts", type=int, default=2)
    p.add_argument("--temperature0", type=float, default=float(os.getenv("ARC_TEMPERATURE0", "1.0")),
                   help="attempt-0 temperature in both thinking and grid phases")
    p.add_argument("--temperature1", type=float, default=float(os.getenv("ARC_TEMPERATURE1", "0.6")),
                   help="attempt-1 temperature in both thinking and grid phases")
    p.add_argument("--strict-pass-order", action=argparse.BooleanOptionalAction, default=False,
                   help="wait for every attempt-0 result before starting attempt-1; may idle the GPU at the tail")
    p.add_argument("--think-budget", type=int, default=int(os.getenv("ARC_THINK_BUDGET", "40000")),
                   help="maximum thinking tokens per attempt (or ARC_THINK_BUDGET)")
    p.add_argument("--think-budget2", type=int, default=int(os.getenv("ARC_THINK_BUDGET2", "0")),
                   help="attempt_2 thinking cap (or ARC_THINK_BUDGET2); 0 = same as --think-budget")
    p.add_argument("--mtp", type=int, default=2, help="MTP draft tokens; used only on EOS-patched checkpoints")
    p.add_argument("--aug2", default=os.getenv("ARC_AUG2", "transpose"), choices=sorted(AUGS),
                   help="view of the task used for attempt_2")
    p.add_argument("--answer-budget", type=int, default=8192)
    p.add_argument("--concurrency", type=int, default=0, help="0 = automatic; positive value overrides policy")
    p.add_argument("--auto-concurrency-policy", choices=("legacy", "kv-safe"), default="legacy",
                   help="legacy uses KV/22k with floor 4; kv-safe budgets the longest selected request")
    p.add_argument("--kv-headroom-fraction", type=float, default=0.20,
                   help="fraction of reported KV capacity reserved by kv-safe policy")
    p.add_argument("--preview-seconds", type=float, default=float(os.getenv("ARC_PREVIEW_SECONDS", "39600")))
    p.add_argument("--preview-tasks", default=os.getenv("ARC_PREVIEW_TASKS", ""), help="comma list (default: all)")
    p.add_argument("--seed-tag", default="kaggle-v1")
    p.add_argument("--server-log", default="/kaggle/working/vllm_server.log")
    a = p.parse_args(argv)
    if a.attempts != 2:
        p.error("Mixed-temperature candidate requires exactly two attempts")
    if not all(0.0 <= t <= 2.0 for t in (a.temperature0, a.temperature1)):
        p.error("Attempt temperatures must be finite numbers in [0, 2]")
    if a.concurrency < 0:
        p.error("--concurrency must be zero or positive")
    if a.think_budget <= 0:
        p.error("--think-budget must be positive")
    if a.think_budget2 < 0:
        p.error("--think-budget2 must be >= 0")
    a.out_dir = Path(a.out_dir)
    a.out_dir.mkdir(parents=True, exist_ok=True)
    t0 = float(os.getenv("ARC_NOTEBOOK_STARTED_MONOTONIC", time.monotonic()))
    rerun = os.getenv("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in ("1", "true", "yes")
    # rerun: Kaggle's 12 h limit also covers container start + model copy-in, which happen before this
    # clock starts at the first cell; keep 45 min of slack
    total = float(os.getenv("ARC_RERUN_SECONDS", "43200")) if rerun else a.preview_seconds
    budget = make_budget(t0, total)
    tasks = json.loads((COMP / ("arc-agi_test_challenges.json" if rerun else "arc-agi_evaluation_challenges.json")).read_text())
    solutions = None if rerun else json.loads((COMP / "arc-agi_evaluation_solutions.json").read_text())
    for t in tasks.values():
        assert all(set(x) == {"input"} for x in t["test"])
    atomic_json(a.out_dir / "submission.json", build_submission(tasks, {}))  # submittable from second 0
    log("MODE", "rerun" if rerun else "preview", "tasks", len(tasks), "budget_s", total)
    model_dir = find_model_dir()
    # patch_lm_head_eos / qwen38-autoround-eosfix write EOSFIX.json next to the weights
    a.patched = (Path(model_dir) / "EOSFIX.json").exists()
    log("MODEL", model_dir, "EOS-patched" if a.patched else "logit_bias EOS mask",
        f"MTP {a.mtp}" if a.patched and a.mtp > 0 else "no MTP")
    log("ATTEMPTS", a.attempts, "| effort", a.effort, "| attempt_1 think cap", think_budget_of(a, 0), "plain",
        "| attempt_2 think cap", think_budget_of(a, 1), a.aug2)
    log("ATTEMPT_TEMPERATURES", "attempt_0", a.temperature0, "attempt_1", a.temperature1,
        "| same temperature for thinking and grid; direct-only FIFO")
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)

    def cost(tid):
        msgs = [{"role": "user", "content": harness_prompt(tasks[tid], k)} for k in range(len(tasks[tid]["test"]))]
        return max(len(tok.apply_chat_template([m], tokenize=True, add_generation_prompt=True, return_dict=True,
                                               enable_thinking=True, reasoning_effort=a.effort)["input_ids"]) for m in msgs)
    selected = [t for t in a.preview_tasks.split(",") if t] if (a.preview_tasks and not rerun) else list(tasks)
    order = sorted(selected, key=cost)
    log("ORDER", len(order), "tasks,", sum(len(tasks[t]["test"]) for t in order), "test inputs,",
        a.attempts * sum(len(tasks[t]["test"]) for t in order), "attempts; first 10 in run order:", order[:10])
    # Flash-Next on 4xL4 leaves ~2-3 GB per GPU after weights: on a GPU-memory startup failure retry with each
    # '||'-separated fallback serving config in turn; any other failure is final (a retry would only repeat it).
    fallbacks = [f.strip() for f in os.getenv("ARC_FALLBACK_SERVER_ARGS", "").split("||") if f.strip()]
    proc = start_server(a, model_dir)
    try:
        for n in range(len(fallbacks) + 1):
            try:
                wait_server(proc, a, budget["stop_new"])
                break
            except RuntimeError as exc:
                cause = str(exc).split("--- log tail ---")[0]
                log("SERVER_ROOT_CAUSE", cause.replace("\n", " | "))
                # a smaller serving config only helps once the weights are in (profiling / KV / CUDA graphs)
                loaded = "Model loading took" in Path(a.server_log).read_text(errors="replace")
                retry_any = os.getenv("ARC_FALLBACK_ANY", "").strip().lower() in ("1", "true", "yes")
                if n == len(fallbacks) or "exited during startup" not in str(exc) or \
                        not (retry_any or (MEM_RE.search(str(exc)) and loaded)):
                    raise
                proc.wait(timeout=60)
                try:                    # engine/worker children of the dead server must release their GPU memory
                    os.killpg(proc.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
                time.sleep(15)
                Path(a.server_log).rename(Path(a.server_log).with_suffix(f".attempt{n + 1}.log"))
                a.extra_server_args = fallbacks[n]
                log("SERVER_FAILED_RETRYING", f"fallback {n + 1}/{len(fallbacks)}:", a.extra_server_args)
                proc = start_server(a, model_dir)
        kv = kv_tokens(a)
        plan = {"policy": "manual" if a.concurrency else a.auto_concurrency_policy,
                "kv_capacity_tokens": kv}
        if a.concurrency:
            conc = a.concurrency
        elif a.auto_concurrency_policy == "kv-safe":
            max_request, skipped = max_selected_request_tokens(tok, tasks, order, a)
            conc, usable = kv_safe_concurrency(kv, max_request, a.max_num_seqs,
                                                a.kv_headroom_fraction)
            plan.update(max_request_tokens=max_request, usable_kv_tokens=usable,
                        skipped_context_requests=skipped, headroom_fraction=a.kv_headroom_fraction)
            if max_request is not None and max_request > usable:
                log("KV_REQUEST_EXCEEDS_BUDGET", "longest", max_request, "usable", usable,
                    "concurrency set to 1; long requests may preempt or still exhaust KV")
        else:
            # Historical heuristic; keep it unchanged until a matched L4 pilot.
            conc = max(4, min(32, (kv or 200000) // 22000))
        log("SERVER_READY", round(time.monotonic() - t0), "s after start; KV tokens", kv,
            "concurrency", conc, "plan", json.dumps(plan))
        rep = asyncio.run(solve(a, tasks, order, solutions, budget, model_dir, conc))
        atomic_json(a.out_dir / "run_report.json", {"mode": "rerun" if rerun else "preview", "kv_tokens": kv,
                                                   "concurrency": conc, "concurrency_plan": plan,
                                                   "args": {k: str(v) for k, v in vars(a).items()},
                                                   "final": rep})
    finally:
        proc.terminate()
        try:
            proc.wait(timeout=60)
        except subprocess.TimeoutExpired:
            proc.kill()


if __name__ == "__main__":
    main()


Writing arc_kaggle_runner.py


In [7]:
import json, os, signal, subprocess, sys, time
from pathlib import Path
t0 = float(os.environ['ARC_NOTEBOOK_STARTED_MONOTONIC'])
rerun = os.getenv('KAGGLE_IS_COMPETITION_RERUN', '').strip().lower() in ('1', 'true', 'yes')
total = float(os.environ['ARC_RERUN_SECONDS']) if rerun else float(os.environ['ARC_PREVIEW_SECONDS'])
cmd = [sys.executable, '-u', 'arc_kaggle_runner.py', '--tp', '4', '--max-model-len', '98304', '--gpu-util', '0.92', '--max-num-seqs', '36', '--concurrency', '32', '--mtp', '3', '--extra-server-args=--kv-cache-dtype fp8 --max-num-batched-tokens 4096 --enable-batch-sharded-sampling']
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                        start_new_session=True)
import threading
def pump():
    for line in proc.stdout:
        print(line, end='', flush=True)
th = threading.Thread(target=pump, daemon=True); th.start()
while proc.poll() is None and time.monotonic() < t0 + total - 240:
    time.sleep(10)
if proc.poll() is None:
    print('WATCHDOG: stopping runner', flush=True)
    try: os.killpg(proc.pid, signal.SIGTERM)
    except ProcessLookupError: pass
    try: proc.wait(timeout=20)
    except subprocess.TimeoutExpired:
        try: os.killpg(proc.pid, signal.SIGKILL)
        except ProcessLookupError: pass
        try: proc.wait(timeout=10)
        except subprocess.TimeoutExpired: pass
th.join(timeout=30)
print('runner exit code', proc.returncode)


09:57:19 MODE preview tasks 120 budget_s 2700.0
09:57:19 MODEL /kaggle/input/datasets/thangnm1/qwen38-27b-arc-rft-v1-nvfp4 EOS-patched MTP 3
09:57:19 ATTEMPTS 2 | effort xhigh | attempt_1 think cap 56000 plain | attempt_2 think cap 32000 shuffle
09:57:19 ATTEMPT_TEMPERATURES attempt_0 0.6 attempt_1 0.6 | same temperature for thinking and grid; direct-only FIFO
09:57:27 ORDER 1 tasks, 1 test inputs, 2 attempts; first 10 in run order: ['e8686506']
09:57:27 SERVER /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model /kaggle/input/datasets/thangnm1/qwen38-27b-arc-rft-v1-nvfp4 --served-model-name qwen38 --host 127.0.0.1 --port 8000 --tensor-parallel-size 4 --language-model-only --reasoning-parser qwen3 --max-model-len 98304 --gpu-memory-utilization 0.92 --max-num-seqs 36 --seed 0 --kv-cache-dtype fp8 --max-num-batched-tokens 4096 --enable-batch-sharded-sampling --no-enable-prefix-caching --speculative-config {"method":"mtp","num_speculative_tokens":3,"use_local_argmax_reduction":t

In [8]:
import json, os
from pathlib import Path
COMP = Path(os.getenv('ARC_COMP_DIR', '/kaggle/input/competitions/arc-prize-2026-arc-agi-2'))
rerun = os.getenv('KAGGLE_IS_COMPETITION_RERUN', '').strip().lower() in ('1', 'true', 'yes')
tasks = json.loads((COMP / ('arc-agi_test_challenges.json' if rerun else 'arc-agi_evaluation_challenges.json')).read_text())
def ok_grid(g):
    return isinstance(g, list) and 1 <= len(g) <= 30 and isinstance(g[0], list) and 1 <= len(g[0]) <= 30 \
        and all(isinstance(r, list) and len(r) == len(g[0]) and all(type(v) is int and 0 <= v <= 9 for v in r) for r in g)
def placeholders():
    return {t: [{'attempt_1': [[0]], 'attempt_2': [[0]]} for _ in task['test']] for t, task in tasks.items()}
sub_path = Path(os.getenv('ARC_SUBMISSION_PATH', '/kaggle/working/submission.json'))
try:
    sub = json.loads(sub_path.read_text())
    assert isinstance(sub, dict) and set(sub) == set(tasks)
    assert all(isinstance(sub[t], list) and len(sub[t]) == len(task['test']) for t, task in tasks.items())
    repaired = 0
    clean = {}
    for t, task in tasks.items():
        clean[t] = []
        for row in sub[t]:
            old = row if isinstance(row, dict) else {}
            new = {}
            for attempt in ('attempt_1', 'attempt_2'):
                grid = old.get(attempt)
                if not ok_grid(grid):
                    grid = [[0]]
                    repaired += 1
                new[attempt] = grid
            clean[t].append(new)
    if clean != sub:
        temp = sub_path.with_name(sub_path.name + '.validator.tmp')
        temp.write_text(json.dumps(clean))
        temp.replace(sub_path)
    print('submission.json valid:', len(clean), 'tasks; repaired attempt slots:', repaired)
except Exception as exc:
    print('submission invalid -> writing placeholders:', exc)
    temp = sub_path.with_name(sub_path.name + '.validator.tmp')
    temp.write_text(json.dumps(placeholders()))
    temp.replace(sub_path)
p = sub_path.with_name('progress.json')
if p.exists():
    print(json.dumps(json.loads(p.read_text()), indent=1))


submission.json valid: 120 tasks; repaired attempt slots: 0
{
 "elapsed_s": 1048.4,
 "work_items": 2,
 "attempts_done": 2,
 "attempt_0_done": 1,
 "attempt_0_grids": 1,
 "grids": 2,
 "closed": 1,
 "think_tokens": 35935,
 "inputs_with_answer": 1,
 "score_all_inputs": 0.58,
 "score_selected_inputs": 100.0,
 "score_started_inputs": 100.0,
 "selected_tasks": 1,
 "selected_inputs": 1,
 "tasks_started": 1,
 "inputs_started": 1,
 "tasks_fully_solved": 1
}
